In [1]:
pip install ultralytics

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import ultralytics
ultralytics.__version__

'8.4.9'

In [4]:
import cv2
import pandas as pd
from ultralytics import YOLO
from tracker import*

model=YOLO('yolov8s.pt')

In [5]:
target_vehicle_ids = [2, 3, 5, 7] 

In [6]:
tracker=Tracker()
count=0

In [7]:
cap=cv2.VideoCapture('6.mp4')

In [8]:
down={}
counter_down=set()

In [ ]:
## first model
while True:    
    ret, frame = cap.read()
    if not ret:
        break

    # Resize to match our line coordinates
    frame = cv2.resize(frame, (1020, 500))

    # 1. Detection (Filtering for Car, Motorcycle, Bus, Truck only)
    results = model.predict(frame, classes=[2, 3, 5, 7], verbose=False)
    a = results[0].boxes.data.detach().cpu().numpy()
    px = pd.DataFrame(a).astype("float")

    list_for_tracker = []
    for index, row in px.iterrows():
        x1, y1, x2, y2 = int(row[0]), int(row[1]), int(row[2]), int(row[3])
        list_for_tracker.append([x1, y1, x2, y2])

    # 2. Congestion Level Logic
    current_count = len(list_for_tracker)
    if current_count > 20:
        congestion_status = "CONGESTION: HIGH"
        status_color = (0, 0, 255) # Red
    else:
        congestion_status = "CONGESTION: NORMAL"
        status_color = (0, 255, 0) # Green

    # 3. Vehicle Tracking & Passing Logic
    bbox_id = tracker.update(list_for_tracker)
    for bbox in bbox_id:
        x3, y3, x4, y4, id = bbox
        cx, cy = (x3 + x4) // 2, (y3 + y4) // 2

        # Virtual Lines (Red = Entry, Blue = Exit)
        if abs(cy - 198) < 8:
            down[id] = cy 
        
        if id in down:
            if abs(cy - 268) < 8:         
                counter_down.add(id) # Successfully passed through both lines

    # 4. Final Output Display
    # Draw the full-length lines
    cv2.line(frame, (0, 198), (1020, 198), (0, 0, 255), 2) # Red line
    cv2.line(frame, (0, 268), (1020, 268), (255, 0, 0), 2) # Blue line

    # Show the two required outputs
    cv2.putText(frame, f"VEHICLES PASSED: {len(counter_down)}", (30, 50), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
    
    cv2.putText(frame, congestion_status, (30, 100), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, status_color, 3)

    cv2.imshow("SolarStat - Traffic Output", frame)
    
    # Press ESC to quit
    if cv2.waitKey(1) & 0xFF == 27:
        break

: 

In [ ]:

while True:    
    ret, frame = cap.read()
    if not ret: break

    frame = cv2.resize(frame, (1020, 500))

    # 1. Detection (Filtering for Car, Motorcycle, Bus, Truck)
    results = model.predict(frame, classes=[2, 3, 5, 7], verbose=False)
    a = results[0].boxes.data.detach().cpu().numpy()
    px = pd.DataFrame(a).astype("float")

    list_for_tracker = []
    for index, row in px.iterrows():
        x1, y1, x2, y2 = int(row[0]), int(row[1]), int(row[2]), int(row[3])
        list_for_tracker.append([x1, y1, x2, y2])

    # 2. FIXED CONGESTION LOGIC (Threshold = 20)
    current_occupancy = len(list_for_tracker)
    if current_occupancy > 20:
        congestion_status = "HIGH"
        status_color = (0, 0, 255) # Red
    else:
        congestion_status = "NORMAL"
        status_color = (0, 255, 0) # Green

    # 3. Vehicle Tracking & Counting
    bbox_id = tracker.update(list_for_tracker)
    for bbox in bbox_id:
        x3, y3, x4, y4, id = bbox
        cy = (y3 + y4) // 2
        
        if abs(cy - 198) < 8:
            down[id] = cy 
        
        if id in down:
            if abs(cy - 268) < 8:         
                counter_down.add(id)

    # 4. TERMINAL OUTPUT (Text in VS Code)
    total_passed = len(counter_down)
    print(f"Passed: {total_passed} | Congestion: {congestion_status} ({current_occupancy} vehicles)")

    # 5. Visual Display
    cv2.putText(frame, f"PASSED: {total_passed}", (30, 50), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    cv2.putText(frame, f"STATUS: {congestion_status}", (30, 100), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, status_color, 3)

    cv2.imshow("SolarStat", frame)
    if cv2.waitKey(1) & 0xFF == 27: break

cap.release()
cv2.destroyAllWindows()



In [ ]:
high_congestion_detected = False
if not cap.isOpened():
    print("Error: Could not open video file. Check the file path!")
    exit() # This stops the script immediately so the report doesn't print
else:
    print("Processing video... (Press ESC to stop)")

while True:    
    ret, frame = cap.read()
    if not ret: break

    frame = cv2.resize(frame, (1020, 500))

    # 1. Detection (Filtering for Car, Motorcycle, Bus, Truck)
    results = model.predict(frame, classes=[2, 3, 5, 7], verbose=False)
    a = results[0].boxes.data.detach().cpu().numpy()
    px = pd.DataFrame(a).astype("float")

    list_for_tracker = []
    for index, row in px.iterrows():
        x1, y1, x2, y2 = int(row[0]), int(row[1]), int(row[2]), int(row[3])
        list_for_tracker.append([x1, y1, x2, y2])

    # 2. FIXED CONGESTION LOGIC (Threshold = 20)
    current_occupancy = len(list_for_tracker)
    if current_occupancy > 25:
        congestion_status = "HIGH"
        status_color = (0, 0, 255) # Red
        high_congestion_detected = True # Log that congestion occurred
    else:
        congestion_status = "NORMAL"
        status_color = (0, 255, 0) # Green

    # 3. Vehicle Tracking & Counting
    bbox_id = tracker.update(list_for_tracker)
    for bbox in bbox_id:
        x3, y3, x4, y4, id = bbox
        cy = (y3 + y4) // 2
        
        # Red Line logic (Entry)
        if abs(cy - 198) < 8:
            down[id] = cy 
        
        # Blue Line logic (Exit)
        if id in down:
            if abs(cy - 268) < 8:         
                counter_down.add(id)

    # 4. Visual Display Improvements
    # Draw the Red and Blue Lines
    cv2.line(frame, (0, 198), (1020, 198), (0, 0, 255), 2) # Entry Line (Red)
    cv2.line(frame, (0, 268), (1020, 268), (255, 0, 0), 2) # Exit Line (Blue)

    total_passed = len(counter_down)
    cv2.putText(frame, f"PASSED: {total_passed}", (30, 50), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    cv2.putText(frame, f"STATUS: {congestion_status}", (30, 100), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, status_color, 3)

    cv2.imshow("SolarStat", frame)
    if cv2.waitKey(1) & 0xFF == 27: break

# --- 5. FINAL TERMINAL OUTPUT (Moved Outside the Loop) ---
cap.release()
cv2.destroyAllWindows()

# Logic to determine the final summary label
final_report_status = "HIGH" if high_congestion_detected else "NORMAL"

print("\n" + "="*40)
print("             FINAL REPORT")
print("="*40)
print(f" TOTAL VEHICLES PASSED    : {len(counter_down)}")
print(f" PEAK CONGESTION OBSERVED : {final_report_status}")
print("="*40)

Processing video... (Press ESC to stop)

             FINAL REPORT
 TOTAL VEHICLES PASSED    : 15
 PEAK CONGESTION OBSERVED : HIGH


In [2]:
import cv2
import pandas as pd
from ultralytics import YOLO
from tracker import *
import firebase_admin
from firebase_admin import credentials, firestore

# Initialize Firebase
cred = credentials.Certificate("D:\c_files\prc\solar\solarstat-c7d5d-5f180bc33d85.json")
firebase_admin.initialize_app(cred)

# Firestore client
db = firestore.client()

# --- INITIALIZATION ---
model = YOLO('yolov8s.pt')
cap = cv2.VideoCapture('6.mp4') 

tracker = Tracker()
counter_down = set()
down = {}
high_congestion_detected = False  

if not cap.isOpened():
    print("Error: Could not open video. Check the file path.")
else:
    print("Processing video... (Press ESC to stop early)")

while True:    
    ret, frame = cap.read()
    if not ret: break

    # Resize to match our line coordinates
    frame = cv2.resize(frame, (1020, 500))

    # 1. Detection (Cars, Motorcycles, Buses, Trucks)
    results = model.predict(frame, classes=[2, 3, 5, 7], verbose=False)
    a = results[0].boxes.data.detach().cpu().numpy()
    px = pd.DataFrame(a).astype("float")

    list_for_tracker = []
    for index, row in px.iterrows():
        x1, y1, x2, y2 = int(row[0]), int(row[1]), int(row[2]), int(row[3])
        list_for_tracker.append([x1, y1, x2, y2])

    # 2. Congestion Logic
    current_occupancy = len(list_for_tracker)
    if current_occupancy > 25:
        congestion_status = "HIGH"
        status_color = (0, 0, 255) # Red
        high_congestion_detected = True
    elif current_occupancy > 10:
        congestion_status = "MEDIUM"
        status_color = (0, 165, 255) # Orange 
    else:
        congestion_status = "NORMAL"
        status_color = (0, 255, 0) # Green

    # 3. Vehicle Tracking & Counting
    bbox_id = tracker.update(list_for_tracker)
    for bbox in bbox_id:
        x3, y3, x4, y4, id = bbox
        cy = (y3 + y4) // 2
        
        # Red Line logic (Entry)
        if abs(cy - 198) < 8:
            down[id] = cy 
        
        # Blue Line logic (Exit)
        if id in down:
            if abs(cy - 268) < 8:         
                counter_down.add(id)

    # 4. Visual Display
    # Draw the Full-Length Counting Lines
    cv2.line(frame, (0, 198), (1020, 198), (0, 0, 255), 2) # Red Line (Entry)
    cv2.line(frame, (0, 268), (1020, 268), (255, 0, 0), 2) # Blue Line (Exit)

    # Display Statistics on screen
    cv2.putText(frame, f"PASSED: {len(counter_down)}", (30, 50), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    cv2.putText(frame, f"STATUS: {congestion_status}", (30, 100), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, status_color, 3)

    cv2.imshow("SolarStat Output", frame)
    if cv2.waitKey(1) & 0xFF == 27: break

# --- 5. FINAL TEXT OUTPUT (Outside the Loop) ---
cap.release()
cv2.destroyAllWindows()

final_status = "HIGH" if high_congestion_detected else "LOW"


print("\n" + "="*40)
print("             FINAL REPORT")
print("="*40)
print(f" TOTAL VEHICLES PASSED    : {len(counter_down)}")
print(f"CONGESTION  : {final_status}")
print("="*40)

<>:9: SyntaxWarning: invalid escape sequence '\c'
<>:9: SyntaxWarning: invalid escape sequence '\c'
C:\Users\amegh\AppData\Local\Temp\ipykernel_22264\192855769.py:9: SyntaxWarning: invalid escape sequence '\c'
  cred = credentials.Certificate("D:\c_files\prc\solar\solarstat-c7d5d-5f180bc33d85.json")


Processing video... (Press ESC to stop early)

             FINAL REPORT
 TOTAL VEHICLES PASSED    : 15
CONGESTION  : LOW


In [1]:
import cv2
import pandas as pd
from ultralytics import YOLO
from tracker import *
import firebase_admin
from firebase_admin import credentials, firestore
from datetime import datetime
from google.cloud.firestore_v1 import FieldFilter

from concurrent.futures import ThreadPoolExecutor

# ---------------- FIREBASE INITIALIZATION ----------------
cred = credentials.Certificate(
    r"D:\c_files\prc\solar\solarstat-c7d5d-0374f53806dc.json"
)

firebase_admin.initialize_app(cred)
db = firestore.client()

# ---------------- LOAD YOLO MODEL ----------------
model = YOLO('yolov8s.pt')

# ---------------- PROCESSING FUNCTION ----------------
def process_region(region, video_url):
    print("\n" + "="*50)
    print(f"Processing Region: {region}")
    print("="*50)

    cap = cv2.VideoCapture(video_url)

    if not cap.isOpened():
        print(f"Error: Could not open video for {region}")
        return

    tracker = Tracker()
    counter_down = set()
    down = {}
    high_congestion_detected = False

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.resize(frame, (1020, 500))

        # -------- YOLO Detection --------
        results = model.predict(frame, classes=[2,3,5,7], verbose=False)
        a = results[0].boxes.data.detach().cpu().numpy()
        px = pd.DataFrame(a).astype("float")

        list_for_tracker = []
        for _, row in px.iterrows():
            x1, y1, x2, y2 = int(row[0]), int(row[1]), int(row[2]), int(row[3])
            list_for_tracker.append([x1, y1, x2, y2])

        # -------- Congestion Logic --------
        if len(list_for_tracker) > 15:
            high_congestion_detected = True

        # -------- Tracking & Counting --------
        bbox_id = tracker.update(list_for_tracker)

        for bbox in bbox_id:
            x3, y3, x4, y4, id = bbox
            cy = (y3 + y4) // 2

            if abs(cy - 198) < 8:
                down[id] = cy

            if id in down and abs(cy - 268) < 8:
                counter_down.add(id)

    cap.release()

    # -------- FINAL RESULTS --------
    vehicle_count = len(counter_down)
    congestion_status = "HIGH" if high_congestion_detected else "LOW"

    print(f"{region} → Vehicles: {vehicle_count}, Congestion: {congestion_status}")

    # -------- UPDATE FIRESTORE --------
    db.collection("traffic_data").document(region).update({
        "vehicleCount": vehicle_count,
        "congestion": congestion_status,
        "processedAt": datetime.now(),
        "needsProcessing": False
    })


# ---------------- FETCH VIDEOS TO PROCESS ----------------
traffic_ref = db.collection("traffic_data")

# Only fetch regions where new video is uploaded
# docs = traffic_ref.where(
#     filter=firestore.FieldFilter("needsProcessing", "==", True)
# ).stream()

docs = traffic_ref.stream()

video_list = []

for doc in docs:
    data = doc.to_dict()
    if "videoUrl" in data:
        video_list.append({
            "region": doc.id,
            "url": data["videoUrl"]
        })

if len(video_list) == 0:
    print("No regions need processing.")
    exit()

print("\nRegions to process:")
for v in video_list:
    print(v)

# ---------------- PARALLEL PROCESSING ----------------
print("\nStarting parallel processing...\n")

with ThreadPoolExecutor(max_workers=4) as executor:
    for item in video_list:
        executor.submit(process_region, item["region"], item["url"])

print("\nAll processing completed.")



Regions to process:
{'region': 'East', 'url': 'https://res.cloudinary.com/drcymgxjb/video/upload/v1771391631/traffic_videos/zohba0qgzbhdfpp57bki.mp4'}
{'region': 'North', 'url': 'https://res.cloudinary.com/drcymgxjb/video/upload/v1771391513/traffic_videos/xvudbbpgo9xgkiwbyfu6.mp4'}
{'region': 'South', 'url': 'https://res.cloudinary.com/drcymgxjb/video/upload/v1771391581/traffic_videos/dwwubkehk8y6zodaf3ze.mp4'}
{'region': 'West', 'url': 'https://res.cloudinary.com/drcymgxjb/video/upload/v1771391723/traffic_videos/hzfnmdvh9cwbmf2xshuy.mp4'}

Starting parallel processing...


Processing Region: East

Processing Region: North

Processing Region: South

Processing Region: West
East → Vehicles: 0, Congestion: LOW
North → Vehicles: 6, Congestion: HIGH
West → Vehicles: 0, Congestion: HIGH
South → Vehicles: 27, Congestion: LOW

All processing completed.


In [ ]:
#automatic video retiving from fb
import cv2
import pandas as pd
from ultralytics import YOLO
from tracker import *
import firebase_admin
from firebase_admin import credentials, firestore
from google.cloud.firestore_v1 import FieldFilter
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor
import time

# ---------------- FIREBASE ----------------
cred = credentials.Certificate(
    r"D:\c_files\prc\solar\solarstat-c7d5d-0374f53806dc.json"
)
import firebase_admin
from firebase_admin import credentials, firestore

# Initialize Firebase only if not already initialized
if not firebase_admin._apps:
    cred = credentials.Certificate(
        r"D:\c_files\prc\solar\solarstat-c7d5d-0374f53806dc.json"
    )
    firebase_admin.initialize_app(cred)

db = firestore.client()



# ---------------- YOLO ----------------
model = YOLO('yolov8s.pt')


# ---------------- PROCESS FUNCTION ----------------
def process_region(region, video_url):
    print(f"\nProcessing {region}")

    cap = cv2.VideoCapture(video_url)
    if not cap.isOpened():
        print(f"Error opening video for {region}")
        return

    tracker = Tracker()
    counter_down = set()
    down = {}
    high_congestion_detected = False

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.resize(frame, (1020, 500))

        results = model.predict(frame, classes=[2,3,5,7], verbose=False)
        a = results[0].boxes.data.detach().cpu().numpy()
        px = pd.DataFrame(a).astype("float")

        list_for_tracker = []
        for _, row in px.iterrows():
            x1, y1, x2, y2 = int(row[0]), int(row[1]), int(row[2]), int(row[3])
            list_for_tracker.append([x1, y1, x2, y2])

        # Congestion logic
        if len(list_for_tracker) > 25:
            high_congestion_detected = True

        # Tracking
        bbox_id = tracker.update(list_for_tracker)
        for bbox in bbox_id:
            x3, y3, x4, y4, id = bbox
            cy = (y3 + y4) // 2

            if abs(cy - 198) < 8:
                down[id] = cy

            if id in down and abs(cy - 268) < 8:
                counter_down.add(id)

    cap.release()

    vehicle_count = len(counter_down)
    congestion = "HIGH" if high_congestion_detected else "LOW"

    print(f"{region} → Vehicles: {vehicle_count}, Congestion: {congestion}")

    # Update Firestore
    db.collection("traffic_data").document(region).update({
        "vehicleCount": vehicle_count,
        "congestion": congestion,
        "processedAt": datetime.now(),
        "needsProcessing": False
    })


# ---------------- CONTINUOUS SERVER LOOP ----------------
print("\nSolarStat Server Started...")

while True:
    try:
        traffic_ref = db.collection("traffic_data")

        # Fetch only regions that need processing
        docs = traffic_ref.where(
            filter=FieldFilter("needsProcessing", "==", True)
        ).stream()

        video_list = []

        for doc in docs:
            data = doc.to_dict()
            if "videoUrl" in data:
                video_list.append({
                    "region": doc.id,
                    "url": data["videoUrl"]
                })

        if len(video_list) > 0:
            print("\nNew videos detected. Starting processing...")

            # Parallel processing
            with ThreadPoolExecutor(max_workers=4) as executor:
                for item in video_list:
                    executor.submit(process_region, item["region"], item["url"])
        else:
            print("No new videos.")

    except Exception as e:
        print("Error:", e)

    # Wait before checking again
    time.sleep(20)   # checks every 20 seconds



SolarStat Server Started...

New videos detected. Starting processing...

Processing East

Processing North

Processing South

Processing West


In [ ]:
import cv2
import pandas as pd
from ultralytics import YOLO
from tracker import *
import firebase_admin
from firebase_admin import credentials, firestore
from datetime import datetime
import time
import torch

# ---------------- GPU SETUP ----------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ---------------- FIREBASE INITIALIZATION ----------------
cred_path = r"D:\c_files\prc\solar\solarstat-c7d5d-0374f53806dc.json"

if not firebase_admin._apps:
    cred = credentials.Certificate(cred_path)
    firebase_admin.initialize_app(cred)

db = firestore.client()

# ---------------- LOAD YOLO ----------------
model = YOLO('yolov8s.pt')
model.to(device)


# ---------------- PROCESS FUNCTION ----------------
def process_region(region, video_url):
    print(f"\nProcessing {region}")

    cap = cv2.VideoCapture(video_url)
    if not cap.isOpened():
        print(f"Error opening video for {region}")
        db.collection("traffic_data").document(region).update({
            "processing": False
        })
        return

    tracker = Tracker()
    counter_down = set()
    down = {}

    total_occupancy = 0
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.resize(frame, (1020, 500))

        # -------- YOLO Detection (GPU) --------
        results = model.predict(
            frame,
            device=device,
            classes=[2,3,5,7],
            verbose=False,
            conf=0.4
        )

        a = results[0].boxes.data.detach().cpu().numpy()
        px = pd.DataFrame(a).astype("float")

        list_for_tracker = []
        for _, row in px.iterrows():
            x1, y1, x2, y2 = int(row[0]), int(row[1]), int(row[2]), int(row[3])
            list_for_tracker.append([x1, y1, x2, y2])

        # -------- Average occupancy --------
        current_occupancy = len(list_for_tracker)
        total_occupancy += current_occupancy
        frame_count += 1

        # -------- Tracking & counting --------
        bbox_id = tracker.update(list_for_tracker)
        for bbox in bbox_id:
            x3, y3, x4, y4, id = bbox
            cy = (y3 + y4) // 2

            if abs(cy - 198) < 8:
                down[id] = cy

            if id in down and abs(cy - 268) < 8:
                counter_down.add(id)

    cap.release()

    # -------- FINAL RESULTS --------
    vehicle_count = len(counter_down)
    avg_occupancy = total_occupancy / frame_count if frame_count > 0 else 0

    # Improved congestion logic
    if avg_occupancy > 20 or vehicle_count > 50:
        congestion = "HIGH"
    elif avg_occupancy > 10 or vehicle_count > 20:
        congestion = "MEDIUM"
    else:
        congestion = "LOW"

    print(f"{region} → Vehicles: {vehicle_count}, AvgDensity: {avg_occupancy:.2f}, Congestion: {congestion}")

    # -------- Update Firestore --------
    db.collection("traffic_data").document(region).update({
        "vehicleCount": vehicle_count,
        "avgOccupancy": avg_occupancy,
        "congestion": congestion,
        "processedAt": datetime.now(),
        "processing": False
    })


# ---------------- REALTIME LISTENER ----------------
def on_snapshot(col_snapshot, changes, read_time):
    for change in changes:
        if change.type.name == 'MODIFIED':
            doc = change.document
            data = doc.to_dict()
            region = doc.id

            if data.get("needsProcessing") == True:
                print(f"\nChange detected → {region}")

                # Lock job immediately
                db.collection("traffic_data").document(region).update({
                    "needsProcessing": False,
                    "processing": True
                })

                process_region(region, data["videoUrl"])


# ---------------- START LISTENER ----------------
print("SolarStat Real-time Listener Started...")

traffic_ref = db.collection("traffic_data")
watch = traffic_ref.on_snapshot(on_snapshot)

while True:
    time.sleep(60)


Using device: cpu
SolarStat Real-time Listener Started...

Change detected → North

Processing North
North → Vehicles: 28, AvgDensity: 8.68, Congestion: MEDIUM

Change detected → West

Processing West
West → Vehicles: 29, AvgDensity: 5.91, Congestion: MEDIUM

Change detected → South

Processing South
South → Vehicles: 14, AvgDensity: 11.73, Congestion: MEDIUM

Change detected → East

Processing East
East → Vehicles: 42, AvgDensity: 9.41, Congestion: MEDIUM

Change detected → East

Processing East


In [ ]:
import cv2
import pandas as pd
from ultralytics import YOLO
from tracker import *
import firebase_admin
from firebase_admin import credentials, firestore
from datetime import datetime
import time
import torch

# ---------------- DEVICE SETUP ----------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ---------------- FIREBASE INITIALIZATION ----------------
cred_path = r"D:\c_files\prc\solar\solarstat-c7d5d-0374f53806dc.json"

if not firebase_admin._apps:
    cred = credentials.Certificate(cred_path)
    firebase_admin.initialize_app(cred)

db = firestore.client()

# ---------------- LOAD YOLO (Faster Model) ----------------
# yolov8n is much faster for CPU systems
model = YOLO('yolov8n.pt')
model.to(device)


# ---------------- PROCESS FUNCTION ----------------
def process_region(region, video_url):
    print(f"\nProcessing {region}")

    cap = cv2.VideoCapture(video_url)
    if not cap.isOpened():
        print(f"Error opening video for {region}")
        db.collection("traffic_data").document(region).update({
            "processing": False
        })
        return

    tracker = Tracker()
    counter_down = set()
    down = {}

    total_occupancy = 0
    frame_count = 0

    frame_skip = 2   # process every 2nd frame for speed
    frame_id = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_id += 1
        if frame_id % frame_skip != 0:
            continue

        frame = cv2.resize(frame, (640, 360))  # smaller size = faster

        # -------- YOLO Detection --------
        results = model.predict(
            frame,
            device=device,
            classes=[2,3,5,7],
            verbose=False,
            conf=0.4
        )

        a = results[0].boxes.data.detach().cpu().numpy()
        px = pd.DataFrame(a).astype("float")

        list_for_tracker = []
        for _, row in px.iterrows():
            x1, y1, x2, y2 = int(row[0]), int(row[1]), int(row[2]), int(row[3])
            list_for_tracker.append([x1, y1, x2, y2])

        # -------- Density calculation --------
        current_occupancy = len(list_for_tracker)
        total_occupancy += current_occupancy
        frame_count += 1

        # -------- Tracking & counting --------
        bbox_id = tracker.update(list_for_tracker)
        for bbox in bbox_id:
            x3, y3, x4, y4, id = bbox
            cy = (y3 + y4) // 2

            if abs(cy - 140) < 8:   # adjusted for new resolution
                down[id] = cy

            if id in down and abs(cy - 190) < 8:
                counter_down.add(id)

    cap.release()

    # -------- FINAL RESULTS --------
    vehicle_count = len(counter_down)
    avg_occupancy = total_occupancy / frame_count if frame_count > 0 else 0

    # -------- Congestion Logic (Flow + Density) --------
    # Adjusted thresholds so HIGH appears realistically
    if vehicle_count > 30 or avg_occupancy > 15:
        congestion = "HIGH"
    elif vehicle_count > 15 or avg_occupancy > 7:
        congestion = "MEDIUM"
    else:
        congestion = "LOW"

    print(f"{region} → Vehicles: {vehicle_count}, AvgDensity: {avg_occupancy:.2f}, Congestion: {congestion}")

    # -------- Update Firestore --------
    db.collection("traffic_data").document(region).update({
        "vehicleCount": vehicle_count,
        "avgOccupancy": avg_occupancy,
        "congestion": congestion,
        "processedAt": datetime.now(),
        "processing": False
    })


# ---------------- REALTIME LISTENER ----------------
def on_snapshot(col_snapshot, changes, read_time):
    for change in changes:
        if change.type.name == 'MODIFIED':
            doc = change.document
            data = doc.to_dict()
            region = doc.id

            if data.get("needsProcessing") == True:
                print(f"\nChange detected → {region}")

                # Lock job immediately
                db.collection("traffic_data").document(region).update({
                    "needsProcessing": False,
                    "processing": True
                })

                process_region(region, data["videoUrl"])


# ---------------- START LISTENER ----------------
print("SolarStat Real-time Listener Started...")

traffic_ref = db.collection("traffic_data")
watch = traffic_ref.on_snapshot(on_snapshot)

while True:
    time.sleep(60)


Using device: cpu
SolarStat Real-time Listener Started...


In [ ]:
import cv2
import pandas as pd
from ultralytics import YOLO
from tracker import *
import firebase_admin
from firebase_admin import credentials, firestore
from datetime import datetime
import time
import torch

# ---------------- DEVICE SETUP ----------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ---------------- FIREBASE INITIALIZATION ----------------
cred_path = r"D:\c_files\prc\solar\solarstat-c7d5d-0374f53806dc.json"

if not firebase_admin._apps:
    cred = credentials.Certificate(cred_path)
    firebase_admin.initialize_app(cred)

db = firestore.client()

# ---------------- LOAD YOLO (FAST FOR CPU) ----------------
model = YOLO('yolov8n.pt')   # Nano model → fastest
model.to(device)


# ---------------- PROCESS FUNCTION ----------------
def process_region(region, video_url):
    print(f"\nProcessing {region}")

    cap = cv2.VideoCapture(video_url)
    if not cap.isOpened():
        print(f"Error opening video for {region}")
        db.collection("traffic_data").document(region).update({
            "processing": False
        })
        return

    tracker = Tracker()
    counter_down = set()
    down = {}

    total_occupancy = 0
    frame_count = 0

    frame_skip = 2
    frame_id = 0

    # Counting lines for 640x360
    entry_line = 140
    exit_line = 190

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_id += 1
        if frame_id % frame_skip != 0:
            continue

        frame = cv2.resize(frame, (640, 360))

        # -------- YOLO Detection --------
        results = model.predict(
            frame,
            device=device,
            classes=[2,3,5,7],
            conf=0.4,
            verbose=False
        )

        detections = results[0].boxes.data.detach().cpu().numpy()

        list_for_tracker = []
        for det in detections:
            x1, y1, x2, y2 = map(int, det[:4])
            list_for_tracker.append([x1, y1, x2, y2])

        # -------- Density calculation --------
        current_occupancy = len(list_for_tracker)
        total_occupancy += current_occupancy
        frame_count += 1

        # -------- Tracking & Counting --------
        bbox_id = tracker.update(list_for_tracker)

        for bbox in bbox_id:
            x3, y3, x4, y4, id = bbox
            cy = (y3 + y4) // 2

            if abs(cy - entry_line) < 8:
                down[id] = cy

            if id in down and abs(cy - exit_line) < 8:
                counter_down.add(id)

    cap.release()

    # -------- FINAL RESULTS --------
    vehicle_count = len(counter_down)
    avg_occupancy = total_occupancy / frame_count if frame_count > 0 else 0

    # -------- FINAL CONGESTION LOGIC --------
    # Tuned for real demo results
    if vehicle_count > 25 or avg_occupancy > 12:
        congestion = "HIGH"
    elif vehicle_count > 10 or avg_occupancy > 6:
        congestion = "MEDIUM"
    else:
        congestion = "LOW"

    print(f"{region} → Vehicles: {vehicle_count}, Density: {avg_occupancy:.2f}, Congestion: {congestion}")

    # -------- Update Firestore --------
    db.collection("traffic_data").document(region).update({
        "vehicleCount": vehicle_count,
        "avgOccupancy": avg_occupancy,
        "congestion": congestion,
        "processedAt": datetime.now(),
        "processing": False
    })


# ---------------- REALTIME LISTENER ----------------
def on_snapshot(col_snapshot, changes, read_time):
    for change in changes:
        if change.type.name == 'MODIFIED':
            doc = change.document
            data = doc.to_dict()
            region = doc.id

            # Prevent duplicate processing
            if data.get("needsProcessing") and not data.get("processing", False):
                print(f"\nChange detected → {region}")

                db.collection("traffic_data").document(region).update({
                    "needsProcessing": False,
                    "processing": True
                })

                process_region(region, data["videoUrl"])


# ---------------- START LISTENER ----------------
print("SolarStat Real-time Listener Started...")

traffic_ref = db.collection("traffic_data")
watch = traffic_ref.on_snapshot(on_snapshot)

# Keep server alive
while True:
    time.sleep(60)

Using device: cpu
SolarStat Real-time Listener Started...


In [3]:
import cv2
import pandas as pd
from ultralytics import YOLO
from tracker import *
import firebase_admin
from firebase_admin import credentials, firestore
from datetime import datetime
import time
import torch
from concurrent.futures import ThreadPoolExecutor
import threading

# ---------------- DEVICE SETUP ----------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ---------------- FIREBASE INITIALIZATION ----------------
cred_path = r"D:\c_files\prc\solar\solarstat-c7d5d-0374f53806dc.json"

if not firebase_admin._apps:
    cred = credentials.Certificate(cred_path)
    firebase_admin.initialize_app(cred)

db = firestore.client()

# ---------------- LOAD YOLO (FAST FOR CPU) ----------------
model = YOLO('yolov8n.pt')   # Nano model → fastest
model.to(device)

# ============================================================================
# PARALLEL PROCESSING SETUP (ThreadPoolExecutor)
# ============================================================================

# Create executor for parallel video processing (max 4 workers for 4 regions)
video_executor = ThreadPoolExecutor(max_workers=4, thread_name_prefix="VideoProcess_")
processing_lock = threading.Lock()  # Prevent duplicate submissions
processing_tasks = {}  # Track ongoing processing tasks by region

print("✅ Parallel Processing Setup: 4 workers ready for concurrent video processing")


# ---------------- PROCESS FUNCTION ----------------
def process_region(region, video_url):
    """Process video for a single region (called in parallel thread)"""
    print(f"\n[WORKER] Processing {region} → Video: {video_url}")

    try:
        cap = cv2.VideoCapture(video_url)
        if not cap.isOpened():
            print(f"[ERROR] Cannot open video for {region}")
            db.collection("traffic_data").document(region).update({
                "processing": False
            })
            return

        tracker = Tracker()
        counter_down = set()
        down = {}

        total_occupancy = 0
        frame_count = 0

        frame_skip = 2
        frame_id = 0

        # Counting lines for 640x360
        entry_line = 140
        exit_line = 190

        print(f"[{region.upper()}] Starting frame analysis...")

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame_id += 1
            if frame_id % frame_skip != 0:
                continue

            frame = cv2.resize(frame, (640, 360))

            # -------- YOLO Detection --------
            results = model.predict(
                frame,
                device=device,
                classes=[2,3,5,7],
                conf=0.4,
                verbose=False
            )

            detections = results[0].boxes.data.detach().cpu().numpy()

            list_for_tracker = []
            for det in detections:
                x1, y1, x2, y2 = map(int, det[:4])
                list_for_tracker.append([x1, y1, x2, y2])

            # -------- Density calculation --------
            current_occupancy = len(list_for_tracker)
            total_occupancy += current_occupancy
            frame_count += 1

            # -------- Tracking & Counting --------
            bbox_id = tracker.update(list_for_tracker)

            for bbox in bbox_id:
                x3, y3, x4, y4, id = bbox
                cy = (y3 + y4) // 2

                if abs(cy - entry_line) < 8:
                    down[id] = cy

                if id in down and abs(cy - exit_line) < 8:
                    counter_down.add(id)

        cap.release()

        # -------- FINAL RESULTS --------
        vehicle_count = len(counter_down)
        avg_occupancy = total_occupancy / frame_count if frame_count > 0 else 0

        # -------- FINAL CONGESTION LOGIC --------
        if vehicle_count > 25 or avg_occupancy > 12:
            congestion = "HIGH"
        elif vehicle_count > 10 or avg_occupancy > 6:
            congestion = "MEDIUM"
        else:
            congestion = "LOW"

        print(f"[{region.upper()}] ✅ Complete → Vehicles: {vehicle_count}, Density: {avg_occupancy:.2f}, Congestion: {congestion}")

        # -------- Update Firestore --------
        db.collection("traffic_data").document(region).update({
            "vehicleCount": vehicle_count,
            "avgOccupancy": avg_occupancy,
            "congestion": congestion,
            "processedAt": datetime.now(),
            "processing": False
        })

    except Exception as e:
        print(f"[ERROR] {region}: {e}")
        db.collection("traffic_data").document(region).update({
            "processing": False
        })

    finally:
        # Remove from tracking
        with processing_lock:
            if region in processing_tasks:
                del processing_tasks[region]


# ============================================================================
# REALTIME LISTENER WITH PARALLEL PROCESSING
# ============================================================================

def on_snapshot(col_snapshot, changes, read_time):
    """Listen for changes and submit to parallel executor"""
    global video_executor, processing_lock, processing_tasks
    
    for change in changes:
        if change.type.name == 'MODIFIED':
            doc = change.document
            data = doc.to_dict()
            region = doc.id

            # Prevent duplicate/concurrent processing of same region
            if data.get("needsProcessing") and not data.get("processing", False):
                print(f"\n[LISTENER] Change detected → {region}")

                with processing_lock:
                    # Only submit if not already processing
                    if region not in processing_tasks:
                        # Set processing flag
                        db.collection("traffic_data").document(region).update({
                            "needsProcessing": False,
                            "processing": True
                        })

                        # Submit to thread pool for parallel processing
                        future = video_executor.submit(process_region, region, data["videoUrl"])
                        processing_tasks[region] = future
                        print(f"[POOL] Submitted {region} to executor (Active tasks: {len(processing_tasks)})")
                    else:
                        print(f"[POOL] {region} already in queue, skipping duplicate")


# ============================================================================
# START PARALLEL LISTENER
# ============================================================================

print("\n" + "="*70)
print("🚀 SolarStat Parallel Listener Started")
print("   Processing Mode: PARALLEL (4 concurrent workers)")
print("   Each region processes independently & simultaneously")
print("="*70 + "\n")

traffic_ref = db.collection("traffic_data")
watch = traffic_ref.on_snapshot(on_snapshot)

# Keep server alive and monitor task queue
print("Listening for video processing requests...")
while True:
    try:
        with processing_lock:
            active_tasks = len(processing_tasks)
            if active_tasks > 0:
                print(f"📊 Active processing tasks: {active_tasks} region(s)")
        time.sleep(5)
    except KeyboardInterrupt:
        print("\n⏹️  Shutting down parallel processor...")
        video_executor.shutdown(wait=True)
        print("✅ All tasks completed")
        break


Using device: cpu
✅ Parallel Processing Setup: 4 workers ready for concurrent video processing

🚀 SolarStat Parallel Listener Started
   Processing Mode: PARALLEL (4 concurrent workers)
   Each region processes independently & simultaneously

Listening for video processing requests...

[LISTENER] Change detected → North

[WORKER] Processing North → Video: https://res.cloudinary.com/drcymgxjb/video/upload/v1772773212/traffic_videos/koraxtkhpj6l5mokwkpe.mp4[POOL] Submitted North to executor (Active tasks: 1)

📊 Active processing tasks: 1 region(s)
[NORTH] Starting frame analysis...
📊 Active processing tasks: 1 region(s)

[LISTENER] Change detected → South

[WORKER] Processing South → Video: https://res.cloudinary.com/drcymgxjb/video/upload/v1772773216/traffic_videos/jdwluf5tiv8a1mjrih9j.mp4[POOL] Submitted South to executor (Active tasks: 2)

[SOUTH] Starting frame analysis...
📊 Active processing tasks: 2 region(s)
📊 Active processing tasks: 2 region(s)

[LISTENER] Change detected → West

In [1]:
print("  🚦 Congestion: Based on vehicle count & LEFT turn ratio")
print("  1️⃣  Priority: Highest LEFT turns goes first")
print("  ⏱️  Timing: Adjusted by LEFT turn count (20-50s GREEN)\n")

# ============================================================================
# REAL-TIME COORDINATOR WITH PRIORITY ROAD SELECTION
# ============================================================================

class RealTimeCoordinator:
    """Coordinator with vehicle distribution & priority road selection"""
    
    def __init__(self, db, regions=['north', 'south', 'east', 'west']):
        self.db = db
        self.regions = regions
        self.running = False
        
    def coordinate_with_priority(self):
        """Coordinate signals using priority road selection"""
        try:
            # Fetch current vehicle counts from all regions
            all_regions_data = {}
            for region in self.regions:
                try:
                    doc = self.db.collection('traffic_data').document(region).get()
                    if doc.exists:
                        all_regions_data[region] = doc.to_dict()
                    else:
                        all_regions_data[region] = {'vehicleCount': 0}
                except Exception as e:
                    print(f"  ⚠️  Error fetching {region}: {e}")
                    all_regions_data[region] = {'vehicleCount': 0}
            
            # Select priority road
            priority_result = select_priority_road(all_regions_data, self.regions)
            priority_road = priority_result['priority_road']
            priority_data = priority_result['priority_road_data']
            all_roads = priority_result['all_roads']
            
            # Display status
            print(f"\n🏆 PRIORITY: {priority_road.upper()} | "
                  f"🔵 {priority_data['left_turns']} LEFT turns | "
                  f"Total: {priority_data['vehicle_count']} vehicles")
            print(f"   ⏱️  {priority_data['timing']['green']}s GREEN | "
                  f"{priority_data['timing']['yellow']}s YELLOW | "
                  f"{priority_data['timing']['red']}s RED")
            
            # Update Firestore for all roads
            for road, road_data in all_roads.items():
                is_priority = (road == priority_road)
                
                # Determine signal state based on priority
                if is_priority:
                    signal_state = 'GREEN'
                    signal_emoji = '🟢'
                    green_time = priority_data['timing']['green']
                else:
                    signal_state = 'RED'
                    signal_emoji = '🔴'
                    green_time = 0
                
                # Distribution info
                distribution = road_data['distribution']
                congestion = road_data['congestion']
                
                # Update Firestore
                self.db.collection('traffic_data').document(road).update({
                    'vehicleCount': road_data['vehicle_count'],
                    'signalState': signal_state,
                    'signalPriority': is_priority,
                    'vehiclesByDirection': {
                        'leftTurns': distribution['LEFT_TURN'],
                        'rightTurns': distribution['RIGHT_TURN'],
                        'straight': distribution['STRAIGHT']
                    },
                    'congestion': congestion['level'],
                    'left_ratio_percent': congestion['left_ratio'],
                    'signalTiming': {
                        'green': priority_data['timing']['green'] if is_priority else 0,
                        'yellow': priority_data['timing']['yellow'],
                        'red': priority_data['timing']['red'],
                        'cycle': priority_data['timing']['cycle']
                    },
                    'timing_reason': priority_data['timing']['selected_reason'],
                    'lastUpdated': datetime.now()
                })
                
                print(f"  {signal_emoji} {road.upper():6} | "
                      f"Vehicle: {road_data['vehicle_count']:2} | "
                      f"L:{distribution['LEFT_TURN']} R:{distribution['RIGHT_TURN']} S:{distribution['STRAIGHT']} | "
                      f"{congestion['emoji']} {congestion['level']}")
        
        except Exception as e:
            print(f"❌ Coordinator error: {e}")
    
    def run_continuous(self):
        """Run coordinator in background"""
        print("🔄 Real-Time Coordinator Starting...")
        self.running = True
        cycle = 0
        
        while self.running:
            try:
                cycle += 1
                print(f"\n📊 Cycle {cycle} - {datetime.now().strftime('%H:%M:%S')}")
                self.coordinate_with_priority()
                time.sleep(5)
            except Exception as e:
                print(f"❌ Error: {e}")
                time.sleep(5)
    
    def start_background(self):
        """Start in background thread"""
        thread = threading.Thread(target=self.run_continuous, daemon=True)
        thread.start()
        return thread

print("✅ RealTimeCoordinator class loaded with priority road selection")

# ============================================================================
# DEMO: VEHICLE DISTRIBUTION & PRIORITY ROAD SELECTION (20 Vehicles Example)
# ============================================================================

print("\n" + "="*70)
print("🚥 SOLARSTAT - VEHICLE DISTRIBUTION & PRIORITY ROAD DEMO")
print("="*70)

# Example scenario: 20 vehicles distributed across 4 roads
example_regions_data = {
    'north': {'vehicleCount': 20},
    'south': {'vehicleCount': 15},
    'east': {'vehicleCount': 25},
    'west': {'vehicleCount': 18}
}

print("\n📊 SCENARIO: 20 vehicles at each intersection")
print("-" * 70)

# Process each region
for region, data in example_regions_data.items():
    total = data['vehicleCount']
    
    # Distribute vehicles
    distribution = distribute_vehicles_by_direction(total)
    congestion = calculate_congestion_with_directions(distribution)
    
    print(f"\n🚗 {region.upper()} ({total} vehicles):")
    print(f"   Distribution: 🔵 {distribution['LEFT_TURN']} LEFT | "
          f"🔴 {distribution['RIGHT_TURN']} RIGHT | "
          f"➡️  {distribution['STRAIGHT']} STRAIGHT")
    print(f"   Congestion: {congestion['emoji']} {congestion['level']} "
          f"({congestion['left_ratio']:.1f}% LEFT turns)")

# Select priority road
priority_result = select_priority_road(example_regions_data)

print("\n" + "="*70)
print("🏆 PRIORITY ROAD SELECTION")
print("="*70)

priority_road = priority_result['priority_road']
priority_data = priority_result['priority_road_data']
all_roads = priority_result['all_roads']

print(f"\n1️⃣  PRIORITY ROAD: {priority_road.upper()}")
print(f"   Reason: {priority_data['timing']['selected_reason']}")
print(f"   🚦 Signal Timing: ")
print(f"      🟢 GREEN:  {priority_data['timing']['green']}s")
print(f"      🟡 YELLOW: {priority_data['timing']['yellow']}s")
print(f"      🔴 RED:    {priority_data['timing']['red']}s")
print(f"      ⏱️  CYCLE:  {priority_data['timing']['cycle']}s")

# Show all roads ranked
print(f"\n📋 ALL ROADS RANKED BY PRIORITY:")
ranked_roads = sorted(all_roads.items(), 
                     key=lambda x: x[1]['priority_score'], 
                     reverse=True)

for rank, (road, data) in enumerate(ranked_roads, 1):
    print(f"   {rank}. {road.upper():6} | {data['vehicle_count']:2} vehicles | "
          f"{data['left_turns']}L {data['right_turns']}R {data['straight']}S | "
          f"Score: {data['priority_score']}")

print("\n" + "="*70)
print("✅ DEMO COMPLETE - Ready for real-time operation")
print("="*70 + "\n")

# ============================================================================
# SECTION 2: SIGNAL TIMING CALCULATOR (LEFT-ONLY)
# ============================================================================

  🚦 Congestion: Based on vehicle count & LEFT turn ratio
  1️⃣  Priority: Highest LEFT turns goes first
  ⏱️  Timing: Adjusted by LEFT turn count (20-50s GREEN)

✅ RealTimeCoordinator class loaded with priority road selection

🚥 SOLARSTAT - VEHICLE DISTRIBUTION & PRIORITY ROAD DEMO

📊 SCENARIO: 20 vehicles at each intersection
----------------------------------------------------------------------


NameError: name 'distribute_vehicles_by_direction' is not defined

In [ ]:
import cv2
import pandas as pd
from ultralytics import YOLO
from tracker import *
import firebase_admin
from firebase_admin import credentials, firestore
from datetime import datetime
import time
import torch

# ---------------- DEVICE SETUP ----------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ---------------- FIREBASE INITIALIZATION ----------------
cred_path = r"D:\c_files\prc\solar\solarstat-c7d5d-0374f53806dc.json"

if not firebase_admin._apps:
    cred = credentials.Certificate(cred_path)
    firebase_admin.initialize_app(cred)

db = firestore.client()

# ---------------- LOAD YOLO (FAST FOR CPU) ----------------
model = YOLO('yolov8n.pt')   # Nano model → fastest
model.to(device)


# ---------------- PROCESS FUNCTION ----------------
def process_region(region, video_url):
    print(f"\nProcessing {region}")

    cap = cv2.VideoCapture(video_url)
    if not cap.isOpened():
        print(f"Error opening video for {region}")
        db.collection("traffic_data").document(region).update({
            "processing": False
        })
        return

    tracker = Tracker()
    counter_down = set()
    down = {}

    total_occupancy = 0
    frame_count = 0

    frame_skip = 2
    frame_id = 0

    # Counting lines for 640x360
    entry_line = 140
    exit_line = 190

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_id += 1
        if frame_id % frame_skip != 0:
            continue

        frame = cv2.resize(frame, (640, 360))

        # -------- YOLO Detection --------
        results = model.predict(
            frame,
            device=device,
            classes=[2,3,5,7],
            conf=0.4,
            verbose=False
        )

        detections = results[0].boxes.data.detach().cpu().numpy()

        list_for_tracker = []
        for det in detections:
            x1, y1, x2, y2 = map(int, det[:4])
            list_for_tracker.append([x1, y1, x2, y2])

        # -------- Density calculation --------
        current_occupancy = len(list_for_tracker)
        total_occupancy += current_occupancy
        frame_count += 1

        # -------- Tracking & Counting --------
        bbox_id = tracker.update(list_for_tracker)

        for bbox in bbox_id:
            x3, y3, x4, y4, id = bbox
            cy = (y3 + y4) // 2

            if abs(cy - entry_line) < 8:
                down[id] = cy

            if id in down and abs(cy - exit_line) < 8:
                counter_down.add(id)

    cap.release()

    # -------- FINAL RESULTS --------
    vehicle_count = len(counter_down)
    avg_occupancy = total_occupancy / frame_count if frame_count > 0 else 0

    # -------- FINAL CONGESTION LOGIC --------
    # Tuned for real demo results
    if vehicle_count > 25 or avg_occupancy > 12:
        congestion = "HIGH"
    elif vehicle_count > 10 or avg_occupancy > 6:
        congestion = "MEDIUM"
    else:
        congestion = "LOW"

    print(f"{region} → Vehicles: {vehicle_count}, Density: {avg_occupancy:.2f}, Congestion: {congestion}")

    # -------- Update Firestore --------
    db.collection("traffic_data").document(region).update({
        "vehicleCount": vehicle_count,
        "avgOccupancy": avg_occupancy,
        "congestion": congestion,
        "processedAt": datetime.now(),
        "processing": False
    })


# ---------------- REALTIME LISTENER ----------------
def on_snapshot(col_snapshot, changes, read_time):
    for change in changes:
        if change.type.name == 'MODIFIED':
            doc = change.document
            data = doc.to_dict()
            region = doc.id

            # Prevent duplicate processing
            if data.get("needsProcessing") and not data.get("processing", False):
                print(f"\nChange detected → {region}")

                db.collection("traffic_data").document(region).update({
                    "needsProcessing": False,
                    "processing": True
                })

                process_region(region, data["videoUrl"])


# ---------------- START LISTENER ----------------
print("SolarStat Real-time Listener Started...")

traffic_ref = db.collection("traffic_data")
watch = traffic_ref.on_snapshot(on_snapshot)

# Keep server alive
while True:
    time.sleep(60)

Using device: cpu
SolarStat Real-time Listener Started...


In [3]:
pip install librosa moviepy ultralytics opencv-python


   ---------------------------------------- 0.0/31.2 MB ? eta -:--:--
   -- ------------------------------------- 1.6/31.2 MB 9.4 MB/s eta 0:00:04
   --- ------------------------------------ 2.9/31.2 MB 8.0 MB/s eta 0:00:04
   ----- ---------------------------------- 4.5/31.2 MB 7.9 MB/s eta 0:00:04
   ------ --------------------------------- 5.2/31.2 MB 6.8 MB/s eta 0:00:04
   -------- ------------------------------- 6.3/31.2 MB 6.3 MB/s eta 0:00:04
   --------- ------------------------------ 7.3/31.2 MB 6.0 MB/s eta 0:00:04
   ---------- ----------------------------- 8.1/31.2 MB 5.5 MB/s eta 0:00:05
   ------------ --------------------------- 9.7/31.2 MB 5.8 MB/s eta 0:00:04
   -------------- ------------------------- 11.3/31.2 MB 6.0 MB/s eta 0:00:04
   ----------------- ---------------------- 13.4/31.2 MB 6.3 MB/s eta 0:00:03
   ------------------ --------------------- 14.4/31.2 MB 6.2 MB/s eta 0:00:03
   ------------------ --------------------- 14.7/31.2 MB 6.2 MB/s eta 0:00:03
  


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
pip install moviepy


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!ffmpeg -version



ffmpeg version 8.0.1-essentials_build-www.gyan.dev Copyright (c) 2000-2025 the FFmpeg developers
built with gcc 15.2.0 (Rev8, Built by MSYS2 project)
configuration: --enable-gpl --enable-version3 --enable-static --disable-w32threads --disable-autodetect --enable-fontconfig --enable-iconv --enable-gnutls --enable-libxml2 --enable-gmp --enable-bzlib --enable-lzma --enable-zlib --enable-libsrt --enable-libssh --enable-libzmq --enable-avisynth --enable-sdl2 --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxvid --enable-libaom --enable-libopenjpeg --enable-libvpx --enable-mediafoundation --enable-libass --enable-libfreetype --enable-libfribidi --enable-libharfbuzz --enable-libvidstab --enable-libvmaf --enable-libzimg --enable-amf --enable-cuda-llvm --enable-cuvid --enable-dxva2 --enable-d3d11va --enable-d3d12va --enable-ffnvcodec --enable-libvpl --enable-nvdec --enable-nvenc --enable-vaapi --enable-openal --enable-libgme --enable-libopenmpt --enable-libopencore-amrwb --enable-

In [3]:
!ffmpeg -i 6.mp4 -q:a 0 -map a audio.wav


ffmpeg version 8.0.1-essentials_build-www.gyan.dev Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 15.2.0 (Rev8, Built by MSYS2 project)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-w32threads --disable-autodetect --enable-fontconfig --enable-iconv --enable-gnutls --enable-libxml2 --enable-gmp --enable-bzlib --enable-lzma --enable-zlib --enable-libsrt --enable-libssh --enable-libzmq --enable-avisynth --enable-sdl2 --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxvid --enable-libaom --enable-libopenjpeg --enable-libvpx --enable-mediafoundation --enable-libass --enable-libfreetype --enable-libfribidi --enable-libharfbuzz --enable-libvidstab --enable-libvmaf --enable-libzimg --enable-amf --enable-cuda-llvm --enable-cuvid --enable-dxva2 --enable-d3d11va --enable-d3d12va --enable-ffnvcodec --enable-libvpl --enable-nvdec --enable-nvenc --enable-vaapi --enable-openal --enable-libgme --enable-libopenmpt --enable-libopencore-amrwb --ena

In [5]:
!pip install librosa numpy scipy




[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
#1
import librosa
import numpy as np

# Load audio
audio_file = "audio.wav"
y, sr = librosa.load(audio_file, sr=None)

# Extract MFCC features
mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

# Calculate energy variation (sirens have strong modulation)
energy = np.mean(np.abs(mfcc[1:]))

print("MFCC Energy:", energy)

# Simple threshold (you can tune this)
if energy > 20:
    print("🚨 Emergency siren detected!")
    emergency_audio = True
else:
    print("No siren detected")
    emergency_audio = False


c:\Users\amegh\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MFCC Energy: 0.0
No siren detected


In [1]:
pip install firebase-admin


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
#2
import firebase_admin
from firebase_admin import credentials, firestore
import requests
import time
import os

# YOLO
from ultralytics import YOLO

# Audio processing
from moviepy.editor import VideoFileClip
import librosa
import numpy as np


# ---------------- FIREBASE SETUP ----------------
cred = credentials.Certificate("serviceAccountKey.json")
firebase_admin.initialize_app(cred)

db = firestore.client()


# ---------------- YOLO MODEL ----------------
model = YOLO("yolov8n.pt")

# Emergency classes available in COCO
EMERGENCY_KEYWORDS = ["ambulance", "fire truck", "police"]


# ---------------- DOWNLOAD VIDEO ----------------
def download_video(url, filename):
    try:
        r = requests.get(url, stream=True, timeout=60)
        with open(filename, "wb") as f:
            for chunk in r.iter_content(1024):
                if chunk:
                    f.write(chunk)
        return True
    except:
        print("Download failed")
        return False


# ---------------- VEHICLE DETECTION ----------------
def detect_emergency_vehicle(video_path):
    try:
        results = model(video_path, stream=True)

        for r in results:
            for box in r.boxes:
                cls = int(box.cls[0])
                label = model.names[cls].lower()

                for keyword in EMERGENCY_KEYWORDS:
                    if keyword in label:
                        print("Emergency vehicle detected:", label)
                        return True

        return False
    except Exception as e:
        print("YOLO error:", e)
        return False


# ---------------- SIREN DETECTION ----------------
def detect_siren(video_path):
    try:
        audio_path = "temp.wav"

        # Extract first 5 seconds
        video = VideoFileClip(video_path).subclip(0, 5)
        video.audio.write_audiofile(audio_path, verbose=False, logger=None)

        y, sr = librosa.load(audio_path, sr=None)

        # FFT
        fft = np.fft.fft(y)
        freqs = np.fft.fftfreq(len(fft), 1/sr)
        mag = np.abs(fft)

        freqs = freqs[:len(freqs)//2]
        mag = mag[:len(mag)//2]

        # Siren range
        mask = (freqs >= 600) & (freqs <= 1800)
        energy = np.mean(mag[mask])

        print("Siren energy:", energy)

        os.remove(audio_path)

        return energy > 50

    except Exception as e:
        print("Audio error:", e)
        return False


# ---------------- COMBINED DECISION ----------------
def process_video(video_path):
    visual = detect_emergency_vehicle(video_path)
    audio = detect_siren(video_path)

    emergency = visual or audio

    print("Final Emergency:", emergency)
    return emergency


# ---------------- FIRESTORE LISTENER ----------------
def on_snapshot(col_snapshot, changes, read_time):
    for change in changes:

        # Only when new document added
        if change.type.name == "ADDED":
            doc = change.document
            data = doc.to_dict()
            doc_id = doc.id

            # Skip if already processed
            if data.get("processed", False):
                continue

            video_url = data.get("videoUrl")

            if not video_url:
                continue

            print("\nNew video detected:", video_url)

            video_file = f"{doc_id}.mp4"

            # Download
            if download_video(video_url, video_file):

                # Process
                emergency = process_video(video_file)

                # Update Firestore
                db.collection("traffic_data").document(doc_id).update({
                    "processed": True,
                    "emergencyDetected": emergency
                })

                # Clean file
                os.remove(video_file)

                print("Updated Firestore")


# ---------------- START LISTENING ----------------
traffic_ref = db.collection("traffic")
watch = traffic_ref.on_snapshot(on_snapshot)

print("Listening for new videos in 'traffic' collection...")

# Keep script running
while True:
    time.sleep(60)


In [18]:
!pip install moviepy



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from moviepy.editor import VideoFileClip
print("MoviePy working")




ModuleNotFoundError: No module named 'moviepy.editor'

In [2]:
!pip show moviepy


Name: moviepy
Version: 2.2.1
Summary: Video editing with Python
Home-page: 
Author: Zulko 2024
Author-email: 
License: MIT License
Location: C:\Users\amegh\AppData\Local\Programs\Python\Python312\Lib\site-packages
Requires: decorator, imageio, imageio_ffmpeg, numpy, pillow, proglog, python-dotenv
Required-by: 


In [19]:
from moviepy.editor import VideoFileClip
print("MoviePy working")


ModuleNotFoundError: No module named 'moviepy.editor'

In [16]:

pip install winget install "FFmpeg (Essentials Build)"
#ffmpeg -version


SyntaxError: invalid syntax (306137971.py, line 1)

In [4]:
!pip uninstall moviepy -y
!pip install moviepy==1.0.3

Found existing installation: moviepy 2.2.1
Uninstalling moviepy-2.2.1:
  Successfully uninstalled moviepy-2.2.1
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for moviepy: filename=moviepy-1.0.3-py3-none-any.whl size=110839 sha256=a608b079d73f0516eec93ce44bd57abea5839f53527ea9c13f1b3e4c34e00a25
  Stored in directory: c:\users\amegh\appdata\local\pip\cache\wheels\df\ba\4b\0917fc0c8833c8ba7016565fc975b74c67bc8610806e930272
Successfully built moviepy

  Attempting uninstall: decorator

    Found existing installation: decorator 5.2.1

    Uninstalling decorator-5.2.1:

   ---------------------------------------- 0/2 [decorator]
      Successfully uninstalled decorator-5.2.1
   ------------------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import firebase_admin
from firebase_admin import credentials, firestore
import requests
import time
import os

# YOLO
from ultralytics import YOLO

# Audio
from moviepy.editor import VideoFileClip
import librosa
import numpy as np


# ---------------- FIREBASE SETUP ----------------
cred = credentials.Certificate(r"D:\c_files\prc\solar\serviceAccountKey.json.json")
firebase_admin.initialize_app(cred)

db = firestore.client()


# ---------------- YOLO MODEL ----------------
model = YOLO("yolov8n.pt")

# Keywords (COCO labels related to emergency)
EMERGENCY_KEYWORDS = ["ambulance", "fire truck", "police"]


# ---------------- DOWNLOAD VIDEO ----------------
def download_video(url, filename):
    try:
        r = requests.get(url, stream=True, timeout=60)
        with open(filename, "wb") as f:
            for chunk in r.iter_content(1024):
                if chunk:
                    f.write(chunk)
        return True
    except Exception as e:
        print("Download failed:", e)
        return False


# ---------------- VEHICLE DETECTION ----------------
def detect_emergency_vehicle(video_path):
    try:
        results = model(video_path, stream=True)

        for r in results:
            for box in r.boxes:
                cls = int(box.cls[0])
                label = model.names[cls].lower()

                for keyword in EMERGENCY_KEYWORDS:
                    if keyword in label:
                        print("Emergency vehicle detected:", label)
                        return True

        return False

    except Exception as e:
        print("YOLO error:", e)
        return False


# ---------------- SIREN DETECTION ----------------
def detect_siren(video_path):
    try:
        audio_path = "temp.wav"

        # Extract first 5 seconds only (faster)
        video = VideoFileClip(video_path).subclip(0, 5)
        video.audio.write_audiofile(audio_path, verbose=False, logger=None)

        y, sr = librosa.load(audio_path, sr=None)

        # FFT
        fft = np.fft.fft(y)
        freqs = np.fft.fftfreq(len(fft), 1/sr)
        mag = np.abs(fft)

        freqs = freqs[:len(freqs)//2]
        mag = mag[:len(mag)//2]

        # Siren frequency range
        mask = (freqs >= 600) & (freqs <= 1800)
        energy = np.mean(mag[mask])

        os.remove(audio_path)

        print("Siren energy:", energy)

        return energy > 50

    except Exception as e:
        print("Audio error:", e)
        return False


# ---------------- FINAL DECISION ----------------
def process_video(video_path):
    visual = detect_emergency_vehicle(video_path)
    audio = detect_siren(video_path)

    emergency = visual or audio
    print("Final Emergency:", emergency)

    return emergency


# ---------------- DOCUMENT LISTENER ----------------
roads = ["north", "south", "east", "west"]

def listen_to_road(road_name):
    doc_ref = db.collection("traffic").document(road_name)

    def on_snapshot(doc_snapshot, changes, read_time):
        for doc in doc_snapshot:
            data = doc.to_dict()

            video_url = data.get("videoUrl")
            processed = data.get("processed", False)

            # Process only when new video uploaded
            if video_url and not processed:
                print(f"\nNew video detected for {road_name}")

                video_file = f"{road_name}.mp4"

                # Download video
                if download_video(video_url, video_file):

                    # Process video
                    emergency = process_video(video_file)

                    # Update Firestore
                    doc_ref.update({
                        "processed": True,
                        "emergencyDetected": emergency
                    })

                    # Remove local file
                    os.remove(video_file)

                    print(f"{road_name} updated → Emergency: {emergency}")

    doc_ref.on_snapshot(on_snapshot)


# ---------------- START LISTENING ----------------
for road in roads:
    listen_to_road(road)

print("Listening for new videos in north, south, east, west...")

# Keep script running
while True:
    time.sleep(60)


Listening for new videos in north, south, east, west...


In [ ]:
import cv2
import pandas as pd
from ultralytics import YOLO
from tracker import *
import firebase_admin
from firebase_admin import credentials, firestore
from datetime import datetime
import time
import torch
import requests
import os
from moviepy.editor import VideoFileClip
import librosa
import numpy as np

# ================= DEVICE =================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ================= FIREBASE =================
cred_path = r"D:\c_files\prc\solar\solarstat-c7d5d-0374f53806dc.json"

if not firebase_admin._apps:
    cred = credentials.Certificate(cred_path)
    firebase_admin.initialize_app(cred)

db = firestore.client()

# ================= YOLO MODELS =================
vehicle_model = YOLO("yolov8n.pt")   # traffic detection
vehicle_model.to(device)

emergency_model = YOLO("yolov8n.pt")  # emergency detection
emergency_model.to(device)

EMERGENCY_KEYWORDS = ["ambulance", "fire truck", "police"]

# ================= DOWNLOAD VIDEO =================
def download_video(url, filename):
    try:
        r = requests.get(url, stream=True, timeout=60)
        with open(filename, "wb") as f:
            for chunk in r.iter_content(1024):
                if chunk:
                    f.write(chunk)
        return True
    except:
        return False


# ================= SIREN DETECTION =================
def detect_siren(video_path):
    try:
        audio_path = "temp.wav"

        video = VideoFileClip(video_path).subclip(0, 5)
        video.audio.write_audiofile(audio_path, verbose=False, logger=None)

        y, sr = librosa.load(audio_path, sr=None)

        fft = np.fft.fft(y)
        freqs = np.fft.fftfreq(len(fft), 1/sr)
        mag = np.abs(fft)

        freqs = freqs[:len(freqs)//2]
        mag = mag[:len(mag)//2]

        mask = (freqs >= 600) & (freqs <= 1800)
        energy = np.mean(mag[mask])

        os.remove(audio_path)

        return energy > 50

    except:
        return False


# ================= MAIN PROCESS =================
def process_video(region, video_url):

    print(f"\nProcessing {region}")

    video_file = f"{region}.mp4"

    # Download video
    if not download_video(video_url, video_file):
        print("Download failed")
        db.collection("traffic_data").document(region).update({"processing": False})
        return

    cap = cv2.VideoCapture(video_file)

    tracker = Tracker()
    counter_down = set()
    down = {}

    total_occupancy = 0
    frame_count = 0
    emergency_detected = False

    frame_skip = 2
    frame_id = 0

    entry_line = 140
    exit_line = 190

    snapshot_saved = False
    snapshot_path = f"{region}.jpg"

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_id += 1
        if frame_id % frame_skip != 0:
            continue

        frame = cv2.resize(frame, (640, 360))

        # ---------- VEHICLE DETECTION ----------
        results = vehicle_model.predict(
            frame,
            device=device,
            classes=[2,3,5,7],
            conf=0.4,
            verbose=False
        )

        detections = results[0].boxes.data.detach().cpu().numpy()

        list_for_tracker = []
        for det in detections:
            x1, y1, x2, y2 = map(int, det[:4])
            list_for_tracker.append([x1, y1, x2, y2])

        # Density
        total_occupancy += len(list_for_tracker)
        frame_count += 1

        # Counting
        bbox_id = tracker.update(list_for_tracker)
        for bbox in bbox_id:
            x3, y3, x4, y4, id = bbox
            cy = (y3 + y4) // 2

            if abs(cy - entry_line) < 8:
                down[id] = cy

            if id in down and abs(cy - exit_line) < 8:
                counter_down.add(id)

        # ---------- EMERGENCY VISUAL DETECTION ----------
        if not emergency_detected:
            for box in results[0].boxes:
                cls = int(box.cls[0])
                label = vehicle_model.names[cls].lower()

                if any(k in label for k in EMERGENCY_KEYWORDS):
                    emergency_detected = True

        # ---------- SAVE SNAPSHOT ----------
        if not snapshot_saved and len(counter_down) > 0:
            cv2.imwrite(snapshot_path, frame)
            snapshot_saved = True

    cap.release()

    # ---------- AUDIO SIREN ----------
    siren_detected = detect_siren(video_file)

    emergency = emergency_detected or siren_detected

    # ---------- RESULTS ----------
    vehicle_count = len(counter_down)
    avg_occupancy = total_occupancy / frame_count if frame_count > 0 else 0

    # Congestion logic
    if vehicle_count > 25 or avg_occupancy > 12:
        congestion = "HIGH"
    elif vehicle_count > 10 or avg_occupancy > 6:
        congestion = "MEDIUM"
    else:
        congestion = "LOW"

    print(f"{region} → Vehicles:{vehicle_count} | Congestion:{congestion} | Emergency:{emergency}")

    # ---------- UPDATE FIRESTORE ----------
    db.collection("traffic_data").document(region).update({
        "vehicleCount": vehicle_count,
        "avgOccupancy": avg_occupancy,
        "congestion": congestion,
        "emergencyDetected": emergency,
        "snapshot": snapshot_path if snapshot_saved else None,
        "processedAt": datetime.now(),
        "processing": False
    })

    # Clean video
    os.remove(video_file)


# ================= LISTENER =================
def on_snapshot(col_snapshot, changes, read_time):
    for change in changes:
        if change.type.name == 'MODIFIED':
            doc = change.document
            data = doc.to_dict()
            region = doc.id

            if data.get("needsProcessing") and not data.get("processing", False):

                print(f"\nTrigger → {region}")

                db.collection("traffic_data").document(region).update({
                    "needsProcessing": False,
                    "processing": True
                })

                process_video(region, data["videoUrl"])


# ================= START SERVER =================
print("SolarStat Unified System Running...")

traffic_ref = db.collection("traffic_data")
watch = traffic_ref.on_snapshot(on_snapshot)

while True:
    time.sleep(60)

Using device: cpu
SolarStat Unified System Running...

Trigger → East

Processing East


Thread-ConsumeBidirectionalStream caught unexpected exception ('Cannot convert to a Firestore Value', np.True_, 'Invalid type', <class 'numpy.bool'>) and will exit.
Traceback (most recent call last):
  File "c:\Users\amegh\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\api_core\bidi.py", line 671, in _thread_main
    self._on_response(response)
  File "c:\Users\amegh\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\firestore_v1\watch.py", line 466, in on_snapshot
    meth(self, proto.target_change)
  File "c:\Users\amegh\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\firestore_v1\watch.py", line 387, in _on_snapshot_target_change_no_change
    self.push(target_change.read_time, target_change.resume_token)
  File "c:\Users\amegh\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\firestore_v1\watch.py", line 567, in push
    self._snapshot_callback(keys, appliedChanges, read_time)
  File "C:\Users\amegh\A

East → Vehicles:29 | Congestion:HIGH | Emergency:True


In [2]:
pip install firebase-admin moviepy librosa numpy requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import firebase_admin
from firebase_admin import credentials, firestore
import requests
import os
from moviepy.editor import VideoFileClip
import librosa
import numpy as np
import time

# ---------------- FIREBASE SETUP ----------------
cred = credentials.Certificate(r"D:\c_files\prc\solar\serviceAccountKey.json.json")
firebase_admin.initialize_app(cred)

db = firestore.client()

print("Listening to traffic_data collection...")

# ---------------- DOWNLOAD VIDEO ----------------
def download_video(url, filename):
    response = requests.get(url)
    with open(filename, "wb") as f:
        f.write(response.content)
    return filename

# ---------------- EXTRACT AUDIO ----------------
def extract_audio(video_path, audio_path):
    video = VideoFileClip(video_path)
    video.audio.write_audiofile(audio_path, logger=None)
    return audio_path

# ---------------- FEATURE EXTRACTION ----------------
def extract_features(audio_file):
    y, sr = librosa.load(audio_file, sr=None)

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_mean = np.mean(mfcc)

    centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    bandwidth = np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))

    return mfcc_mean, centroid, bandwidth

# ---------------- CLASSIFICATION ----------------
def classify_siren(features):
    _, centroid, bandwidth = features

    # No siren
    if centroid < 1500:
        return "No emergency vehicle"

    # Pattern-based classification
    if bandwidth < 2000:
        return "Ambulance"
    elif 2000 <= bandwidth < 3500:
        return "Police"
    else:
        return "Fire Truck"

# ---------------- PROCESS DOCUMENT ----------------
def process_document(doc_id, data):
    try:
        video_url = data.get("videoUrl")
        if not video_url:
            return

        print(f"\nProcessing {doc_id}...")

        video_file = f"{doc_id}.mp4"
        audio_file = f"{doc_id}.wav"

        # Download
        print("Downloading video...")
        download_video(video_url, video_file)

        # Extract audio
        print("Extracting audio...")
        extract_audio(video_file, audio_file)

        # Analyze
        print("Analyzing siren...")
        features = extract_features(audio_file)
        result = classify_siren(features)

        print(f"{doc_id} Result:", result)

        # Update Firestore
        db.collection("traffic_data").document(doc_id).update({
            "processed": True,
            "emergency": result,
            "processed_at": firestore.SERVER_TIMESTAMP
        })

        # Cleanup
        os.remove(video_file)
        os.remove(audio_file)

    except Exception as e:
        print(f"Error in {doc_id}:", e)

# ---------------- LISTENER ----------------
def on_snapshot(col_snapshot, changes, read_time):
    for change in changes:
        doc = change.document
        data = doc.to_dict()

        # Trigger when document is added or modified
        if change.type.name in ["ADDED", "MODIFIED"]:
            if data.get("processed") == False:
                process_document(doc.id, data)

# ---------------- START LISTENING ----------------
collection_ref = db.collection("traffic_data")
collection_ref.on_snapshot(on_snapshot)

# Keep running
while True:
    time.sleep(60)

Listening to traffic_data collection...


In [ ]:
## 📦 Prepare a Training Dataset (YOLO format)

To train a custom model you need:

1. **Images** (e.g. `dataset/images/train/` and `dataset/images/val/`).
2. **Labels** in YOLO text format (one `.txt` per image) under `dataset/labels/train/` and `dataset/labels/val/`.

Each label line is: `class_id x_center y_center width height` (all normalized 0–1).

Example label file for an image `0001.jpg`:
```
0 0.532 0.462 0.320 0.210
0 0.740 0.380 0.220 0.160
``` 
(where `0` is the class index for your object type).

> You can create these `.txt` files using a labeling tool (like LabelImg, Roboflow, etc.) and export to YOLO format.

In [ ]:
# Create a minimal data config YAML for YOLO training
# Update these paths to match where your dataset is stored.

import os

data_yaml = {
    'train': 'dataset/images/train',
    'val':   'dataset/images/val',
    'nc': 1,  # number of classes (update as needed)
    'names': ['vehicle']  # class names (update for your classes)
}

os.makedirs('dataset/images/train', exist_ok=True)
os.makedirs('dataset/images/val', exist_ok=True)
os.makedirs('dataset/labels/train', exist_ok=True)
os.makedirs('dataset/labels/val', exist_ok=True)

import yaml
with open('data.yaml', 'w') as f:
    yaml.dump(data_yaml, f)

print('Created data.yaml (edit it for your dataset paths / classes).')
print(open('data.yaml').read())

## ▶️ Train a Custom YOLO Model

Once you have a dataset and `data.yaml` ready, run training like this.

(Adjust `epochs`, `batch`, `imgsz`, and the base model to match your hardware.)

In [ ]:
from ultralytics import YOLO

# Choose a base pretrained model (yolov8n/yolov8s/yolov8m etc.)
model = YOLO('yolov8n.pt')

# Train (this will create weights under runs/train/)
model.train(
    data='data.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    workers=4,
    name='custom_traffic',
)

# After training you can use the best weights for inference:
# best_path = 'runs/train/custom_traffic/weights/best.pt'
# model = YOLO(best_path)
